# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anasYaha/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Je compare trois methodes du toolkit sur le meme jeu de 5 honest features defini en w03
(top_content_impression_share et is_cannibalized restent hors features -- ce sont le label et
son input direct) :

- **Logistic Regression** -- reference lineaire, interpretable, donne des coefficients directs.
- **Decision Tree** -- regle lisible, comparable en esprit a mon baseline heuristique w04.
- **Random Forest** -- deja teste en w03 pour le leakage check ; je reconfirme son AUC honnete,
  mais cette fois sous un vrai split groupe par client (voir section 2).

**Unite d'analyse** : je passe du grain (client, query, content) de w03 au grain (client, query),
qui est le niveau ou `is_cannibalized` est reellement defini (valeur constante pour toutes les
pages d'une meme query) et le niveau auquel mon `baseline_action_score` de w04 classe deja.
Comparer a deux grains differents fausserait Precision@K.


In [3]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass, duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
QUERY_ID_COL = 'query_hash_id'  # confirmed in w03_data_contract.ipynb

# Same (client, query, content) feature frame as w03/w04, restricted to the March-active slice
cannib = con.sql(f"""
    WITH per_query AS (
        SELECT client_hash_id, {QUERY_ID_COL} AS query_id, content_hash_id,
               ANY_VALUE(content_visible_query_count)  AS content_visible_query_count,
               ANY_VALUE(rare_impressions_share)        AS rare_impressions_share,
               ANY_VALUE(anonymized_impressions_share)  AS anonymized_impressions_share,
               SUM(impressions_90d)                     AS content_impressions_for_query
        FROM {TABLES['fact_query_90d']}
        WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM {TABLES['fact_daily_mar']})
        GROUP BY 1, 2, 3
    ),
    query_totals AS (
        SELECT client_hash_id, query_id,
               COUNT(DISTINCT content_hash_id)   AS competing_content_count,
               SUM(content_impressions_for_query) AS query_total_impressions,
               MAX(content_impressions_for_query) AS top_content_impressions
        FROM per_query
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.query_id, p.content_hash_id,
           p.content_visible_query_count, p.rare_impressions_share, p.anonymized_impressions_share,
           qt.competing_content_count, qt.query_total_impressions,
           qt.top_content_impressions / NULLIF(qt.query_total_impressions, 0) AS top_content_impression_share
    FROM per_query p
    JOIN query_totals qt USING (client_hash_id, query_id)
""").df()

cannib['is_cannibalized'] = (cannib['top_content_impression_share'] < 0.70).astype(int)

# Collapse (client, query, content) -> (client, query): aggregate the per-content ANY_VALUE
# fields with mean/max across competing pages (honest -- knowable pre-decision, never touches
# top_content_impression_share). This is the grain my w04 baseline scores at.
query_level = (
    cannib.groupby(['client_hash_id', 'query_id'], as_index=False)
    .agg(
        competing_content_count=('competing_content_count', 'first'),
        query_total_impressions=('query_total_impressions', 'first'),
        top_content_impression_share=('top_content_impression_share', 'first'),  # context only
        is_cannibalized=('is_cannibalized', 'first'),
        mean_content_visible_query_count=('content_visible_query_count', 'mean'),
        max_content_visible_query_count=('content_visible_query_count', 'max'),
        mean_rare_impressions_share=('rare_impressions_share', 'mean'),
        mean_anonymized_impressions_share=('anonymized_impressions_share', 'mean'),
    )
)
print(f'{len(query_level):,} (client, query) pairs')
query_level.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1,129,695 (client, query) pairs


,client_hash_id,query_id,competing_content_count,query_total_impressions,top_content_impression_share,is_cannibalized,mean_content_visible_query_count,max_content_visible_query_count,mean_rare_impressions_share,mean_anonymized_impressions_share
0,client_08a6a72ff48e62c0,query_000160888182a46a,1,78.0,1.000000,0,3.0,3,0.024111,0.838578
1,client_08a6a72ff48e62c0,query_00027a3c3893bd13,8,200.0,0.175000,1,4.5,9,0.253755,0.591251
2,client_08a6a72ff48e62c0,query_0002ad8569888867,2,113.0,0.513274,1,14.5,22,0.003749,0.813867
3,client_08a6a72ff48e62c0,query_0003c9af4a5cbd52,1,11.0,1.000000,0,37.0,37,0.039460,0.683180
4,client_08a6a72ff48e62c0,query_0007159b7fe2ce91,5,826.0,0.635593,1,222.8,607,0.016490,0.550025


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Groupe par `client_hash_id`, pas aleatoire ligne a ligne. w03 stratifiait sur y mais sans
grouper par client -- un meme client pouvait apparaitre en train ET en test, ce qui fuit
(comportements de fragmentation propres a un client). Ici : `GroupShuffleSplit` sur
`client_hash_id`, test_size=0.25, random_state=42 (meme seed que w03/w04 pour rester
comparable), pour mesurer ce qui compte vraiment : est-ce que ca generalise a des clients
jamais vus a l'entrainement.


In [4]:
from sklearn.model_selection import GroupShuffleSplit

honest_features = [
    'competing_content_count', 'query_total_impressions',
    'mean_content_visible_query_count', 'max_content_visible_query_count',
    'mean_rare_impressions_share', 'mean_anonymized_impressions_share',
]
model_df = query_level.dropna(subset=honest_features + ['is_cannibalized']).copy()
model_df['log_query_total_impressions'] = np.log1p(model_df['query_total_impressions'])

feature_cols = [c for c in honest_features if c != 'query_total_impressions'] + ['log_query_total_impressions']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print(f'Train: {len(train_df):,} rows, {train_df["client_hash_id"].nunique()} clients')
print(f'Test:  {len(test_df):,} rows, {test_df["client_hash_id"].nunique()} clients')
print(f'Clients overlapping train/test: {len(overlap)} (expect 0)')


Train: 708,647 rows, 32 clients
Test:  421,048 rows, 11 clients
Clients overlapping train/test: 0 (expect 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Meme split, meme test set pour les 3 modeles ET pour le baseline (recalcule sur le test set
uniquement, formule w04 inchangee). Deux metriques : AUC (calibration globale) et
Precision@50 (aligne sur le pipeline de reference FlyRank qui vise ~0.24 -> 0.74 baseline
vs modele).


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_tr, y_tr = train_df[feature_cols], train_df['is_cannibalized']
X_te, y_te = test_df[feature_cols], test_df['is_cannibalized']

models = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'decision_tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

# Baseline recomputed on the test set only (same formula as w04, no refitting)
base_test = test_df.copy()
def pct_rank(s): return s.rank(pct=True, method='average')
base_test['baseline_score'] = (
    pct_rank(np.log1p(base_test['query_total_impressions'])) * pct_rank(base_test['competing_content_count'])
)

results = [{
    'model': 'baseline (w04)',
    'auc': round(roc_auc_score(y_te, base_test['baseline_score']), 3),
    'precision_at_50': round(precision_at_k(y_te, base_test['baseline_score'].values), 3),
}]

for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    results.append({
        'model': name,
        'auc': round(roc_auc_score(y_te, proba), 3),
        'precision_at_50': round(precision_at_k(y_te, proba), 3),
    })

results_df = pd.DataFrame(results)
print(results_df)
best_model_name = results_df.iloc[1:].sort_values('precision_at_50', ascending=False).iloc[0]['model']
print(f'\nBest model on Precision@50: {best_model_name}')


                 model    auc  precision_at_50
0       baseline (w04)  0.940             0.86
1  logistic_regression  0.977             0.96
2        decision_tree  0.987             1.00
3        random_forest  0.987             1.00

Best model on Precision@50: decision_tree


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

J'inspecte ou le meilleur modele se trompe (faux positifs / faux negatifs dans son top-50)
et sur quoi il s'appuie (feature_importances_ ou coefficients). Un faux positif ici = query
jugee prioritaire par le modele alors qu'elle n'est pas vraiment cannibalisee (une page
domine deja) ; un faux negatif = cannibalisation reelle que le modele sous-score.

**TODO apres execution** : ecrire ici 2-3 phrases sur ce que revelent les faux positifs /
faux negatifs (ex. queries a fort volume mais faible fragmentation reelle, ou l'inverse) et
sur quelle feature domine le modele.


In [6]:
best_clf = models[best_model_name]
proba = best_clf.predict_proba(X_te)[:, 1]
test_df_scored = test_df.copy()
test_df_scored['model_score'] = proba
test_df_scored['predicted_top50'] = test_df_scored['model_score'].rank(ascending=False, method='first') <= 50

top50 = test_df_scored[test_df_scored['predicted_top50']]
false_positives = top50[top50['is_cannibalized'] == 0]
print(f'Top-50 modele : {len(false_positives)} faux positifs sur 50')
print(false_positives[['client_hash_id', 'query_id', 'competing_content_count',
                        'top_content_impression_share', 'model_score']].head(10))

missed = test_df_scored[(test_df_scored['is_cannibalized'] == 1)].sort_values('model_score').head(10)
print('\n10 vrais cannibalises les moins bien scores par le modele :')
print(missed[['client_hash_id', 'query_id', 'competing_content_count',
              'top_content_impression_share', 'model_score']])

if hasattr(best_clf, 'feature_importances_'):
    imp = pd.Series(best_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print('\nFeature importances:\n', imp)
elif hasattr(best_clf, 'coef_'):
    coef = pd.Series(best_clf.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
    print('\nCoefficients:\n', coef)


Top-50 modele : 0 faux positifs sur 50
Empty DataFrame
Columns: [client_hash_id, query_id, competing_content_count, top_content_impression_share, model_score]
Index: []

10 vrais cannibalises les moins bien scores par le modele :
                 client_hash_id                query_id  \
307908  client_23a62021009f63c4  query_cb00c5421e1cc5ae   
989666  client_e547b89c05043229  query_b0d72b13e5c03e80   
214377  client_23a62021009f63c4  query_50e9a335e01e3433   
58925   client_0fa64a184f18a4a0  query_da6810ab4928617b   
214524  client_23a62021009f63c4  query_511ad2eaf95a238a   
214661  client_23a62021009f63c4  query_514779adc96f57ec   
292582  client_23a62021009f63c4  query_b743f48f5a980161   
155911  client_23a62021009f63c4  query_049533cdbf0cf623   
990600  client_e547b89c05043229  query_b2834160a8d3d5f0   
308157  client_23a62021009f63c4  query_cb5787500643caba   

        competing_content_count  top_content_impression_share  model_score  
307908                        2            

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.